# 01 — Data Understanding
**FlightIQ — AI Travel Price Intelligence**

MIC AIML Department Recruitment Challenge  
Track: Data Science & Visualization — AI Travel Analyst

---
**Goal**: Inspect the raw dataset as-is. No cleaning, no EDA, no modelling.

In [1]:
import sys
import os
import pandas as pd

# Make src importable
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'src'))
from data_loader import load_raw_data, get_data_summary

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Load Dataset

In [2]:
df = load_raw_data()
print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape}")

Dataset loaded successfully.
Shape: (100000, 18)


## 2. Basic Info

In [3]:
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nColumn Names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

Rows    : 100,000
Columns : 18

Column Names:
   1. Flight_ID
   2. Airline
   3. Source
   4. Destination
   5. Departure_Date
   6. Departure_Time
   7. Arrival_Time
   8. Duration
   9. Total_Stops
  10. Distance_km
  11. Travel_Class
  12. Days_Before_Departure
  13. Season
  14. Weekday
  15. Aircraft_Type
  16. Booking_Channel
  17. Passenger_Count
  18. Price


In [4]:
df.head(5)

,Flight_ID,Airline,Source,Destination,Departure_Date,Departure_Time,Arrival_Time,Duration,Total_Stops,Distance_km,Travel_Class,Days_Before_Departure,Season,Weekday,Aircraft_Type,Booking_Channel,Passenger_Count,Price
0,FL133547,Indigo,Hyderabad,Ahmedabad,2025-08-02,8:10 PM,21:50,1.67,0,910.5,Economy,43,Monsoon,Saturday,Boeing 737,Mobile App,5,5181.56
1,FL133458,AirAsia India,Pune,Mumbai,2025-06-05,12:10,12:55,0h 45m,non-stop,150,Economy,45,Monsoon,Thursday,ATR 72,Website,3,2000
2,FL173851,Qatar Airways,Sydney,PNQ,2025-02-11,5:05 PM,7:53 AM,14.80,1 stop,10168.1,Economy,3,NaN,Tuesday,Boeing 777,Mobile App,3,174762.69
3,FL101953,AirAsia India,Bangalore Airport,BOM,2025-04-19,07:05,10:16 AM,3h 11m,1,868.3,Economy,33,Summer,Saturday,Airbus A320,Mobile App,1,2846.09
4,FL162887,British Airways,Sydney,Goa,2026-07-05,10:40 PM,12:36,13.93,non-stop,10478.0,Economy,6,Monsoon,Sunday,Airbus A320,Website,2,145331.64


## 3. Data Types

In [5]:
print("Raw dtypes from pandas:")
print(df.dtypes)
print("\nNote: Many columns read as 'object' due to mixed value formats in the CSV.")

Raw dtypes from pandas:
Flight_ID                object
Airline                  object
Source                   object
Destination              object
Departure_Date           object
Departure_Time           object
Arrival_Time             object
Duration                 object
Total_Stops              object
Distance_km              object
Travel_Class             object
Days_Before_Departure    object
Season                   object
Weekday                  object
Aircraft_Type            object
Booking_Channel          object
Passenger_Count          object
Price                    object
dtype: object

Note: Many columns read as 'object' due to mixed value formats in the CSV.


## 4. Missing Values

In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(f"Total missing values: {missing.sum():,}")
print()
print(missing_df.to_string())

Total missing values: 82,966

                       Missing Count  Missing %
Price                           5053       5.05
Duration                        5042       5.04
Booking_Channel                 4962       4.96
Departure_Date                  4927       4.93
Distance_km                     4923       4.92
Source                          4915       4.92
Total_Stops                     4905       4.90
Destination                     4881       4.88
Airline                         4880       4.88
Departure_Time                  4868       4.87
Travel_Class                    4863       4.86
Aircraft_Type                   4852       4.85
Passenger_Count                 4826       4.83
Arrival_Time                    4789       4.79
Days_Before_Departure           4784       4.78
Season                          4776       4.78
Weekday                         4720       4.72


## 5. Duplicate Rows

In [7]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count:,} ({dup_count/len(df)*100:.2f}%)")

Duplicate rows: 1,961 (1.96%)


## 6. Unique Values Per Column

In [8]:
unique_df = pd.DataFrame({
    'Column': df.columns,
    'Unique Values': [df[col].nunique() for col in df.columns],
    'Sample Values': [str(df[col].dropna().unique()[:3].tolist()) for col in df.columns]
}).set_index('Column')
print(unique_df.to_string())

                       Unique Values                                 Sample Values
Column                                                                            
Flight_ID                      98039          ['FL133547', 'FL133458', 'FL173851']
Airline                           39  ['Indigo', 'AirAsia India', 'Qatar Airways']
Source                            54               ['Hyderabad', 'Pune', 'Sydney']
Destination                       54                ['Ahmedabad', 'Mumbai', 'PNQ']
Departure_Date                   730    ['2025-08-02', '2025-06-05', '2025-02-11']
Departure_Time                   576               ['8:10 PM', '12:10', '5:05 PM']
Arrival_Time                    2880                 ['21:50', '12:55', '7:53 AM']
Duration                        5111                   ['1.67', '0h 45m', '14.80']
Total_Stops                        6                   ['0', 'non-stop', '1 stop']
Distance_km                    59484                   ['910.5', '150', '10168.1']
Trav

## 7. Column Classification

In [9]:
# Based on actual inspection of the dataset
column_classification = {
    'identifier': ['Flight_ID'],
    'categorical': ['Airline', 'Source', 'Destination', 'Travel_Class', 'Season',
                    'Weekday', 'Aircraft_Type', 'Booking_Channel', 'Total_Stops'],
    'numerical': ['Distance_km', 'Days_Before_Departure', 'Passenger_Count', 'Price'],
    'date': ['Departure_Date'],
    'time': ['Departure_Time', 'Arrival_Time'],
    'duration': ['Duration'],
    'stops_related': ['Total_Stops'],
    'route_related': ['Source', 'Destination', 'Distance_km'],
    'target': ['Price']
}

print("Column Classification (based on actual dataset inspection):")
for category, cols in column_classification.items():
    print(f"\n  {category.upper():20}: {cols}")

Column Classification (based on actual dataset inspection):

  IDENTIFIER          : ['Flight_ID']

  CATEGORICAL         : ['Airline', 'Source', 'Destination', 'Travel_Class', 'Season', 'Weekday', 'Aircraft_Type', 'Booking_Channel', 'Total_Stops']

  NUMERICAL           : ['Distance_km', 'Days_Before_Departure', 'Passenger_Count', 'Price']

  DATE                : ['Departure_Date']

  TIME                : ['Departure_Time', 'Arrival_Time']

  DURATION            : ['Duration']

  STOPS_RELATED       : ['Total_Stops']

  ROUTE_RELATED       : ['Source', 'Destination', 'Distance_km']

  TARGET              : ['Price']


## 8. Target Variable

In [10]:
target = 'Price'
print(f"Target Variable : {target}")
print(f"Dtype (raw)     : {df[target].dtype}")
print(f"Missing values  : {df[target].isnull().sum()}")
print(f"Unique values   : {df[target].nunique():,}")
print(f"\nSample values:")
print(df[target].dropna().head(10).tolist())

Target Variable : Price
Dtype (raw)     : object
Missing values  : 5053
Unique values   : 78,725

Sample values:
['5181.56', '2000', '174762.69', '2846.09', '145331.64', '190059.64', '6458.88', '28599.41', '147645.15', '24660.74']


## 9. Summary

In [11]:
print("=" * 50)
print("PART 0 — DATA UNDERSTANDING SUMMARY")
print("=" * 50)
print(f"Dataset Shape    : {df.shape}")
print(f"Target Variable  : Price")
print(f"Total Columns    : {df.shape[1]}")
print(f"Total Missing    : {df.isnull().sum().sum():,}")
print(f"Duplicate Rows   : {df.duplicated().sum():,}")
print("=" * 50)
print("NOTE: Duration column has mixed formats (e.g., '1.67' and '0h 45m').")
print("NOTE: Total_Stops has mixed formats (e.g., '0', '1', 'non-stop').")
print("NOTE: Numerical columns (Price, Distance_km, etc.) read as object due to mixed types.")
print("NOTE: Cleaning and type-casting will be done in Part 1.")

PART 0 — DATA UNDERSTANDING SUMMARY
Dataset Shape    : (100000, 18)
Target Variable  : Price
Total Columns    : 18
Total Missing    : 82,966


Duplicate Rows   : 1,961
NOTE: Duration column has mixed formats (e.g., '1.67' and '0h 45m').
NOTE: Total_Stops has mixed formats (e.g., '0', '1', 'non-stop').
NOTE: Numerical columns (Price, Distance_km, etc.) read as object due to mixed types.
NOTE: Cleaning and type-casting will be done in Part 1.
